# 속성별 분리 문장 감성분석 테스트

문장 분리 모델이 만든 `속성명 + 분리 문장`을 감성 모델에 바로 넣어보는 노트북

- 수동 테스트: 원하는 속성과 문장을 직접 입력
- DB 테스트: `review_aspect_sentences`에서 문장을 읽어서 분석
- DB 저장은 하지 않음

In [ ]:
# 프로젝트 위치와 감성 모델 준비
import os
from pathlib import Path

import pandas as pd
import pymysql
import torch
from dotenv import load_dotenv
from IPython.display import display
from pymysql.cursors import DictCursor
from transformers import AutoModelForSequenceClassification, AutoTokenizer

root_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    path for path in root_candidates
    if (path / 'models' / 'sentiment' / 'config.json').is_file()
)
MODEL_DIR = ROOT / 'models' / 'sentiment'
MAX_LENGTH = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

print('프로젝트:', ROOT)
print('사용 장치:', device)
print('감성 라벨:', model.config.id2label)

In [ ]:
# 여러 문장을 한 번에 분석하는 함수
@torch.inference_mode()
def predict_sentiments(rows):
    if not rows:
        return pd.DataFrame()

    model_inputs = [
        f"[속성] {row['속성']} [문장] {row['문장']}"
        for row in rows
    ]
    encoded = tokenizer(
        model_inputs,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    )
    encoded = {name: value.to(device) for name, value in encoded.items()}

    probabilities = torch.softmax(model(**encoded).logits, dim=-1)
    predicted_ids = probabilities.argmax(dim=-1)

    results = []
    for row, scores, predicted_id in zip(rows, probabilities, predicted_ids):
        label_id = int(predicted_id)
        results.append({
            **row,
            '예측 감성': model.config.id2label[label_id],
            '신뢰도': round(float(scores[label_id]), 4),
            '긍정 확률': round(float(scores[0]), 4),
            '부정 확률': round(float(scores[1]), 4),
            '중립 확률': round(float(scores[2]), 4),
        })

    return pd.DataFrame(results)

## 1. 원하는 문장을 직접 입력해서 테스트

`속성`에는 모델 학습 속성명을 입력

예: `지속력/유지력`, `보습력/수분감`, `밀착력/접착력`

In [ ]:
# 이 목록만 수정하면 바로 다른 문장을 테스트 가능
test_rows = [
    {'속성': '보습력/수분감', '문장': '촉촉하게 잘 발려요'},
    {'속성': '지속력/유지력', '문장': '금방 지워져요'},
    {'속성': '커버력', '문장': '커버를 위해 사용하실 분은 비추입니다'},
    {'속성': '자극성', '문장': '바르고 나니 피부가 따가워요'},
]

manual_result = predict_sentiments(test_rows)
display(manual_result)

## 2. DB에 저장된 분리 문장으로 테스트

아래 `REVIEW_ID`에 확인할 내부 리뷰 ID를 입력

값을 `None`으로 두면 문장 분리 결과 앞에서부터 `LIMIT`개 조회

In [ ]:
# DB 연결
load_dotenv(ROOT / '.env')
if not all(os.getenv(name) for name in ('host', 'ID', 'PW', 'DBName')):
    load_dotenv(ROOT.parent / 'aspect_sentence_split' / '.env')

connection = pymysql.connect(
    host=os.getenv('host'),
    user=os.getenv('ID'),
    password=os.getenv('PW'),
    db=os.getenv('DBName'),
    charset='utf8mb4',
    cursorclass=DictCursor,
)
print('DB 연결 완료')

In [ ]:
# 특정 리뷰만 보고 싶으면 숫자 입력, 전체 표본은 None
REVIEW_ID = None
LIMIT = 30

if REVIEW_ID is None:
    sql = '''
        SELECT
            ras.aspect_sentence_id,
            ras.review_id,
            ras.model_attribute_name,
            ras.separated_sentence
        FROM review_aspect_sentences ras
        ORDER BY ras.aspect_sentence_id
        LIMIT %s
    '''
    params = (LIMIT,)
else:
    sql = '''
        SELECT
            ras.aspect_sentence_id,
            ras.review_id,
            ras.model_attribute_name,
            ras.separated_sentence
        FROM review_aspect_sentences ras
        WHERE ras.review_id = %s
        ORDER BY ras.aspect_sentence_id
    '''
    params = (REVIEW_ID,)

with connection.cursor() as cursor:
    cursor.execute(sql, params)
    db_rows = cursor.fetchall()

print('조회 문장 수:', len(db_rows))

In [ ]:
# DB 컬럼을 테스트 함수 형식으로 바꾼 뒤 감성분석
db_test_rows = [
    {
        'aspect_sentence_id': row['aspect_sentence_id'],
        'review_id': row['review_id'],
        '속성': row['model_attribute_name'],
        '문장': row['separated_sentence'],
    }
    for row in db_rows
]

db_result = predict_sentiments(db_test_rows)
display(db_result)

In [ ]:
# 테스트가 끝나면 DB 연결 정리
connection.close()
print('DB 연결 종료')